# Tiền xử lý dữ liệu Messi: `matches` và `appearances`

Notebook này tạo hai bảng từ **StatsBomb Open Data** để kết hợp với `messi_all_goals.csv` (bộ dữ liệu bàn thắng đã tải sẵn từ Kaggle).

**Phạm vi xử lý**
- La Liga: Barcelona, mùa 2004/05–2020/21.
- Ligue 1: Paris Saint-Germain, mùa 2021/22–2022/23.
- Chỉ giữ những trận Messi thực sự ra sân.
- `matches`: thông tin trận đấu và tỷ số theo góc nhìn của câu lạc bộ Messi.
- `appearances`: thông tin Messi có đá chính hay không, số phút thi đấu, vị trí bắt đầu và lý do kết thúc.

**Lưu ý:** số phút được ước tính theo mốc phút nguyên trong dữ liệu StatsBomb, giới hạn tối đa 90 phút và không tính bù giờ. `age` là tuổi tính theo năm bắt đầu mùa giải (`năm bắt đầu mùa - 1987`), không phải tuổi chính xác tại ngày diễn ra trận đấu.

## 1. Cài đặt thư viện
Chạy ô dưới đây nếu môi trường Jupyter chưa có các thư viện cần thiết.

In [ ]:
%pip install -q statsbombpy pandas

## 2. Import thư viện và thiết lập cấu hình

Không tắt toàn bộ cảnh báo của Python, vì cảnh báo có thể giúp phát hiện vấn đề trong quá trình tải hoặc xử lý dữ liệu.

In [ ]:
import sqlite3
from pathlib import Path

import pandas as pd
from statsbombpy import sb

MESSI_PLAYER_ID = 5503
MESSI_BIRTH_YEAR = 1987
REGULATION_MINUTES = 90

LEAGUE_CONFIGS = [
    {
        "competition_id": 11,
        "competition_name": "La Liga",
        "club": "Barcelona",
        "first_season_start_year": 2004,
        "last_season_start_year": 2020,
    },
    {
        "competition_id": 7,
        "competition_name": "Ligue 1",
        "club": "Paris Saint-Germain",
        "first_season_start_year": 2021,
        "last_season_start_year": 2022,
    },
]

OUTPUT_DIRECTORY = Path("data/processed")
OUTPUT_DIRECTORY.mkdir(parents=True, exist_ok=True)

## 3. Xác định các mùa giải cần tải

Lấy danh sách giải đấu và mùa giải mà StatsBomb Open Data cung cấp, sau đó lọc theo giải đấu và khoảng mùa đã xác định ở trên.

In [ ]:
def get_season_start_year(season_name):
    """Lấy năm bắt đầu từ tên mùa giải, ví dụ '2011/2012' -> 2011."""
    return int(str(season_name).split("/")[0])


def find_seasons_to_download():
    """Trả về danh sách các mùa giải nằm trong phạm vi dự án."""
    all_competitions = sb.competitions()
    seasons_to_download = []

    for league in LEAGUE_CONFIGS:
        league_seasons = all_competitions[
            all_competitions["competition_id"] == league["competition_id"]
        ]

        for _, season_row in league_seasons.iterrows():
            season_start_year = get_season_start_year(season_row["season_name"])

            if not (
                league["first_season_start_year"]
                <= season_start_year
                <= league["last_season_start_year"]
            ):
                continue

            seasons_to_download.append({
                "competition_id": league["competition_id"],
                "competition_name": league["competition_name"],
                "club": league["club"],
                "season_id": int(season_row["season_id"]),
                "season_name": season_row["season_name"],
                "season_start_year": season_start_year,
            })

    seasons_to_download.sort(
        key=lambda season: (season["season_start_year"], season["competition_id"])
    )
    return seasons_to_download


seasons = find_seasons_to_download()
pd.DataFrame(seasons)

## 4. Tính số phút thi đấu và lấy thông tin lần ra sân của Messi

StatsBomb ghi các khoảng thời gian chơi ở từng vị trí (*position spells*). Nếu cầu thủ đổi vị trí trong trận, có thể có nhiều khoảng; ta dùng mốc bắt đầu đầu tiên và mốc kết thúc cuối cùng. Trường hợp `to` bị để trống được hiểu là chơi đến hết 90 phút chính thức.

In [ ]:
def clock_string_to_minute(clock_string):
    """Chuyển chuỗi thời gian '67:31' thành phút nguyên 67."""
    return int(str(clock_string).split(":")[0])


def compute_minutes_from_positions(position_spells):
    """Ước tính phút thi đấu, bỏ bù giờ và giới hạn trong khoảng 1–90 phút.

    Danh sách position_spells rỗng nghĩa là cầu thủ không ra sân.
    Nếu vào sân trong thời gian bù giờ, quy ước tính tối thiểu 1 phút.
    """
    if not position_spells:
        return 0

    first_spell = position_spells[0]
    last_spell = position_spells[-1]

    start_minute = clock_string_to_minute(first_spell["from"])
    end_minute = (
        REGULATION_MINUTES
        if last_spell["to"] is None
        else clock_string_to_minute(last_spell["to"])
    )

    # Bỏ phần bù giờ: mọi mốc sau phút 90 được quy về phút 90.
    start_minute = min(start_minute, REGULATION_MINUTES)
    end_minute = min(end_minute, REGULATION_MINUTES)

    # Giữ quy ước của bộ dữ liệu: người vào sân rất muộn vẫn tính ít nhất 1 phút.
    return max(end_minute - start_minute, 1)


def get_messi_appearance(match_id, club):
    """Lấy thông tin ra sân của Messi; trả về None nếu không ra sân."""
    lineups_by_team = sb.lineups(match_id=match_id)

    if club not in lineups_by_team:
        return None

    club_lineup = lineups_by_team[club]
    messi_rows = club_lineup[club_lineup["player_id"] == MESSI_PLAYER_ID]

    if messi_rows.empty:
        return None

    position_spells = messi_rows.iloc[0]["positions"]
    minutes_played = compute_minutes_from_positions(position_spells)

    if minutes_played == 0:
        return None

    first_spell = position_spells[0]
    last_spell = position_spells[-1]

    return {
        "match_id": int(match_id),
        "started": int(first_spell["start_reason"] == "Starting XI"),
        "minutes_played": minutes_played,
        "starting_position": first_spell["position"],
        "end_reason": last_spell["end_reason"],
    }

## 5. Chuẩn hóa thông tin trận đấu

Các cột `goals_for` và `goals_against` luôn được tính theo góc nhìn của đội Messi: bàn đội Messi ghi và bàn đối thủ ghi. `venue` cho biết đội Messi đá sân nhà (`home`) hay sân khách (`away`).

In [ ]:
def build_match_row(statsbomb_match, season):
    """Chuyển một trận StatsBomb thành một dòng trong bảng matches."""
    club = season["club"]
    is_home_match = statsbomb_match["home_team"] == club

    if is_home_match:
        opponent = statsbomb_match["away_team"]
        venue = "home"
        goals_for = statsbomb_match["home_score"]
        goals_against = statsbomb_match["away_score"]
    else:
        opponent = statsbomb_match["home_team"]
        venue = "away"
        goals_for = statsbomb_match["away_score"]
        goals_against = statsbomb_match["home_score"]

    return {
        "match_id": int(statsbomb_match["match_id"]),
        "date": statsbomb_match["match_date"],
        "season": season["season_name"],
        "age": season["season_start_year"] - MESSI_BIRTH_YEAR,
        "competition": season["competition_name"],
        "match_week": int(statsbomb_match["match_week"]),
        "club": club,
        "opponent": opponent,
        "venue": venue,
        "home_score": int(statsbomb_match["home_score"]),
        "away_score": int(statsbomb_match["away_score"]),
        "goals_for": int(goals_for),
        "goals_against": int(goals_against),
    }

## 6. Tải và ghép dữ liệu thành hai bảng

Với từng mùa, tải danh sách trận; chỉ xử lý những trận có Barcelona hoặc PSG tham gia theo cấu hình. Sau đó kiểm tra đội hình để xác nhận Messi có ra sân. Mỗi trận được thêm vào cả hai bảng để có thể ghép bằng `match_id`.

In [ ]:
def build_tables():
    """Tải dữ liệu StatsBomb và trả về hai DataFrame: matches, appearances."""
    match_rows = []
    appearance_rows = []

    for season in seasons:
        statsbomb_matches = sb.matches(
            competition_id=season["competition_id"],
            season_id=season["season_id"],
        )

        appearances_this_season = 0

        for _, statsbomb_match in statsbomb_matches.iterrows():
            teams_in_match = (
                statsbomb_match["home_team"],
                statsbomb_match["away_team"],
            )
            if season["club"] not in teams_in_match:
                continue

            match_id = int(statsbomb_match["match_id"])
            appearance = get_messi_appearance(match_id, season["club"])

            # Không thêm trận nếu Messi có tên trong dữ liệu đội hình nhưng không ra sân.
            if appearance is None:
                continue

            match_rows.append(build_match_row(statsbomb_match, season))
            appearance_rows.append(appearance)
            appearances_this_season += 1

        print(
            f"{season['competition_name']} {season['season_name']}: "
            f"{appearances_this_season} lần ra sân"
        )

    matches_table = pd.DataFrame(match_rows)
    appearances_table = pd.DataFrame(appearance_rows)

    if matches_table.empty:
        raise ValueError(
            "Không tạo được dữ liệu trận đấu. Hãy kiểm tra kết nối hoặc phạm vi StatsBomb."
        )

    matches_table = matches_table.sort_values("date").reset_index(drop=True)
    appearances_table = matches_table[["match_id"]].merge(
        appearances_table, on="match_id", how="left", validate="one_to_one"
    )

    return matches_table, appearances_table


matches_table, appearances_table = build_tables()
display(matches_table.head())
display(appearances_table.head())

## 7. Kiểm tra chất lượng dữ liệu

Các kiểm tra dưới đây đảm bảo `match_id` không bị trùng, hai bảng có cùng số trận và mỗi trận trong `matches` có đúng một dòng tương ứng trong `appearances`. Đây là kiểm tra cấu trúc; vẫn nên đối chiếu một số trận với nguồn gốc nếu cần xác nhận độ chính xác tuyệt đối.

In [ ]:
assert matches_table["match_id"].is_unique, "match_id bị trùng trong matches"
assert appearances_table["match_id"].is_unique, "match_id bị trùng trong appearances"
assert len(matches_table) == len(appearances_table), "Hai bảng khác số dòng"
assert matches_table["match_id"].equals(appearances_table["match_id"]), (
    "Thứ tự hoặc danh sách match_id giữa hai bảng không khớp"
)
assert matches_table[["date", "season", "club", "opponent"]].notna().all().all()
assert appearances_table[["started", "minutes_played"]].notna().all().all()
assert appearances_table["started"].isin([0, 1]).all()
assert appearances_table["minutes_played"].between(1, 90).all()

print(f"Số trận: {len(matches_table)}")
print(f"Số trận Messi đá chính: {int(appearances_table['started'].sum())}")
print(f"Tổng số phút ước tính: {int(appearances_table['minutes_played'].sum())}")
print("Các kiểm tra cấu trúc đều đạt.")

## 8. Lưu dữ liệu thành CSV và SQLite

Notebook lưu hai tệp CSV và một cơ sở dữ liệu SQLite trong thư mục `data/processed/`. Nếu chạy lại, các bảng SQLite cùng tên sẽ được thay thế bằng kết quả mới.

In [ ]:
matches_path = OUTPUT_DIRECTORY / "matches.csv"
appearances_path = OUTPUT_DIRECTORY / "appearances.csv"
database_path = OUTPUT_DIRECTORY / "messi.db"

matches_table.to_csv(matches_path, index=False)
appearances_table.to_csv(appearances_path, index=False)

with sqlite3.connect(database_path) as connection:
    matches_table.to_sql("matches", connection, if_exists="replace", index=False)
    appearances_table.to_sql("appearances", connection, if_exists="replace", index=False)

print(f"Đã lưu: {matches_path}")
print(f"Đã lưu: {appearances_path}")
print(f"Đã lưu: {database_path}")

## 9. Tổng hợp theo mùa giải

Bảng tổng hợp giúp kiểm tra số lần ra sân, số trận đá chính và số phút theo từng mùa. Số phút là kết quả ước tính theo quy ước ở phần 4.

In [ ]:
season_summary = (
    matches_table.merge(
        appearances_table, on="match_id", validate="one_to_one"
    )
    .groupby(["season", "age", "competition"], as_index=False)
    .agg(
        appearances=("match_id", "count"),
        starts=("started", "sum"),
        minutes=("minutes_played", "sum"),
    )
)

display(season_summary)
print(f"Tổng số trận: {len(matches_table)}")
print(f"Tổng số phút ước tính: {int(appearances_table['minutes_played'].sum())}")

## 10. Đọc và kiểm tra bộ bàn thắng lấy từ Kaggle (tùy chọn)

Đặt `messi_all_goals.csv` cùng thư mục với notebook hoặc sửa đường dẫn bên dưới. Bộ này là dữ liệu riêng về bàn thắng; không dùng nó để tạo hai bảng `matches` và `appearances`. Có thể dùng để kiểm tra sơ bộ tên cột, số dòng và dữ liệu thiếu trước khi ghép/phân tích chung.

In [ ]:
GOALS_CSV_PATH = Path("messi_all_goals.csv")

if GOALS_CSV_PATH.exists():
    goals_table = pd.read_csv(GOALS_CSV_PATH)
    print("Kích thước bộ bàn thắng:", goals_table.shape)
    print("Các cột:", goals_table.columns.tolist())
    display(goals_table.head())
    display(
        goals_table.isna().sum()
        .rename("missing_values")
        .to_frame()
        .query("missing_values > 0")
    )
else:
    print(
        f"Chưa tìm thấy {GOALS_CSV_PATH}. "
        "Đặt tệp CSV cạnh notebook để chạy phần kiểm tra tùy chọn."
    )